# Constructing Documents for RAG

This tutorial is part of **AviosIT Tutorials**, a technical learning platform covering data science, machine learning, AI engineering, and software development.

It is a continuation of the [Preparing Quran Dataset for RAG](https://aviosit.com/tutorials/preparing-quran-dataset-for-rag) tutorial, where we prepared a structured Quran dataset for downstream retrieval tasks.

In this tutorial, we will transform the prepared dataset into documents suitable for a Retrieval-Augmented Generation (RAG) pipeline. Each document will contain a text field and metadata that can be used during retrieval and source attribution.

This process is part of [ChatQuran: A Multilingual Quran RAG Application](https://aviosit.com/projects/chatquran), a multilingual RAG application for Quranic question answering. The concepts introduced here establish the document representation needed for subsequent stages, including embedding, indexing, and retrieval.

The example `Parquet` file contains synthetic placeholder text for demonstration only. It is not actual Quranic text or translations. The same document-construction pattern can be applied to the complete dataset when it is available locally.

Each document contains an ID, text, and metadata. Keeping these fields explicit makes documents easier to identify, filter, retrieve, and cite later in a RAG pipeline.

## Learning Objectives

By the end of this tutorial, you will be able to:

- Load a prepared Quran dataset.
- Construct documents from individual ayah records in multiple languages.
- Separate document text from metadata.
- Validate and inspect the resulting documents.

**Prerequisite:** Complete the [Preparing Quran Dataset for RAG](https://aviosit.com/tutorials/preparing-quran-dataset-for-rag) tutorial, or use the example dataset provided in this repository.

## 1. Load the Prepared Dataset

Load the example Parquet dataset included with this tutorial. The path resolution below supports running the notebook from the repository root or from its own `notebooks/` directory.

In [1]:
from pathlib import Path

import pandas as pd

candidate_paths = [
    Path("constructing-documents-for-rag/data/processed/quran.parquet"),
    Path("../data/processed/quran.parquet"),
]

dataset_path = next((path for path in candidate_paths if path.exists()), None)

if dataset_path is None:
    raise FileNotFoundError(
        "Could not find quran-example.parquet. Run this notebook from the "
        "repository root or from constructing-documents-for-rag/notebooks/."
    )

quran_df = pd.read_parquet(dataset_path)

print(f"Loaded dataset: {dataset_path}")
print(f"Rows: {len(quran_df):,}")
print(f"Columns: {list(quran_df.columns)}")
display(quran_df)

Loaded dataset: ../data/processed/quran.parquet
Rows: 6,236
Columns: ['surah', 'ayah', 'arabic', 'english', 'indonesian']


,surah,ayah,arabic,english,indonesian
0,1,1,بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ,"In the name of Allah, the Entirely Merciful, t...",Dengan menyebut nama Allah Yang Maha Pemurah l...
1,1,2,ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَـٰلَمِينَ,"[All] praise is [due] to Allah, Lord of the wo...","Segala puji bagi Allah, Tuhan semesta alam."
2,1,3,ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ,"The Entirely Merciful, the Especially Merciful,",Maha Pemurah lagi Maha Penyayang.
3,1,4,مَـٰلِكِ يَوْمِ ٱلدِّينِ,Sovereign of the Day of Recompense.,Yang menguasai di Hari Pembalasan.
4,1,5,إِيَّاكَ نَعْبُدُ وَإِيَّاكَ نَسْتَعِينُ,It is You we worship and You we ask for help.,"Hanya Engkaulah yang kami sembah, dan hanya ke..."
...,...,...,...,...,...
6231,114,2,مَلِكِ ٱلنَّاسِ,The Sovereign of mankind.,Raja manusia.
6232,114,3,إِلَـٰهِ ٱلنَّاسِ,"The God of mankind,",Sembahan manusia.
6233,114,4,مِن شَرِّ ٱلْوَسْوَاسِ ٱلْخَنَّاسِ,From the evil of the retreating whisperer -,Dari kejahatan (bisikan) syaitan yang biasa be...
6234,114,5,ٱلَّذِى يُوَسْوِسُ فِى صُدُورِ ٱلنَّاسِ,Who whispers [evil] into the breasts of mankind -,yang membisikkan (kejahatan) ke dalam dada man...


The prepared dataset has one row per ayah, with separate columns for Arabic, English, and Indonesian text. The example contains only one synthetic row, so the next step should produce three documents: one per language.

## 2. Construct Documents

For this tutorial, a document is represented as a Python dictionary with three fields:

- `id`: a stable identifier combining the surah, ayah, and language.
- `text`: the text that a retrieval system will search.
- `metadata`: structured information used for filtering, tracing, and source attribution.

We create one document per language for every row in the prepared dataset.

In [2]:
language_columns = {
    "ar": "arabic",
    "en": "english",
    "id": "indonesian",
}

documents = []

for row in quran_df.itertuples(index=False):
    for language, column in language_columns.items():
        document = {
            "id": f"{row.surah}:{row.ayah}:{language}",
            "text": getattr(row, column),
            "metadata": {
                "ayah_id": f"{row.surah}:{row.ayah}",
                "surah": int(row.surah),
                "ayah": int(row.ayah),
                "language": language,
            },
        }
        documents.append(document)

print(f"Constructed {len(documents):,} documents from {len(quran_df):,} rows.")

Constructed 18,708 documents from 6,236 rows.


The document count should be three times the number of rows because each ayah is represented in three languages. This assumes all three language columns are present and populated.

In [3]:
expected_document_count = len(quran_df) * len(language_columns)
assert len(documents) == expected_document_count

required_fields = {"id", "text", "metadata"}
assert all(required_fields.issubset(document) for document in documents)

print("Document count and required fields validated.")

Document count and required fields validated.


## 3. Inspect the Results

Inspect a few documents to see how the source row is represented in the new structure. The ID distinguishes languages, while `ayah_id` links the language-specific documents to the same ayah.

In [4]:
for document in documents[:3]:
    print(document)
    print()

{'id': '1:1:ar', 'text': 'بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ', 'metadata': {'ayah_id': '1:1', 'surah': 1, 'ayah': 1, 'language': 'ar'}}

{'id': '1:1:en', 'text': 'In the name of Allah, the Entirely Merciful, the Especially Merciful.', 'metadata': {'ayah_id': '1:1', 'surah': 1, 'ayah': 1, 'language': 'en'}}

{'id': '1:1:id', 'text': 'Dengan menyebut nama Allah Yang Maha Pemurah lagi Maha Penyayang.', 'metadata': {'ayah_id': '1:1', 'surah': 1, 'ayah': 1, 'language': 'id'}}



We can also retrieve all language-specific documents associated with a particular ayah by filtering its `ayah_id` metadata.

In [5]:
example_ayah_id = f"{quran_df.iloc[0]['surah']}:{quran_df.iloc[0]['ayah']}"

ayah_documents = [
    document
    for document in documents
    if document["metadata"]["ayah_id"] == example_ayah_id
]

print(f"Documents for ayah {example_ayah_id}: {len(ayah_documents)}")
for document in ayah_documents:
    print(f"{document['id']}: {document['text']}")

Documents for ayah 1:1: 3
1:1:ar: بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ
1:1:en: In the name of Allah, the Entirely Merciful, the Especially Merciful.
1:1:id: Dengan menyebut nama Allah Yang Maha Pemurah lagi Maha Penyayang.


## Conclusion

In this tutorial, we transformed a structured Quran dataset into documents suitable for a Retrieval-Augmented Generation (RAG) pipeline. Each document contains a unique identifier, text, and metadata that preserve its relationship to the original ayah and language.

This document representation provides a foundation for the next stages of [ChatQuran: A Multilingual Quran RAG Application](https://aviosit.com/projects/chatquran), a multilingual RAG application for Quranic question answering.

The next step is to generate embeddings from the document text and prepare the documents for indexing and retrieval.

This tutorial is part of **AviosIT Tutorials**, where individual technical concepts are explored and connected to practical AI engineering projects.